In [6]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet')

train_kf_oof = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_kf_oof_train2_index_fixed.parquet')

print(train.shape, val.shape, train_kf_oof.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(616293, 35) (133316, 35) (616293, 40)


In [2]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np
import optuna

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

c:\Users\soohok402\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# ============================================================
# 2. 종속변수 설정
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [7]:
train.head()

,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,imerg_rain_1h,imerg_rain_6h,imerg_rain_24h,sfc_rain_1h,sfc_rain_6h,sfc_rain_24h,radar_reflectivity_mean,radar_reflectivity_p95,radar_reflectivity_ge20_fraction,cat_area,wsarea,mean_elevation,mean_slope,urban_ratio,forest_ratio,stream_order,target_maxrise_6h,event_id,wl_diff_1h,wl_diff_3h,wl_diff_6h,wl_diff_12h,wl_diff_24h,imerg_int_1to6h,imerg_int_6to24h,sfc_int_1to6h,sfc_int_6to24h
row_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
TR_000222241,2100-01-01,ST0001,0.29,0.29,0.28,0.28,0.28,0.27,0.000,0.000,0.000,0.0,0.0,0.0,12.162,25.274,0.1445,2.8605,37.0279,124.37,9.6147,11.1141,52.4597,1.9592,0.00,1,0.00,0.01,0.01,0.01,0.02,0.000,0.000000,0.0,0.0
TR_000288748,2100-01-01,ST0002,0.77,0.76,0.77,0.77,0.76,0.77,0.000,0.000,0.000,0.0,0.0,0.0,0.912,12.060,0.0000,3.9564,42.8163,180.09,16.5687,2.5710,88.0622,1.8000,0.00,1,0.01,0.00,0.00,0.01,0.00,0.000,0.000000,0.0,0.0
TR_000737576,2100-01-01,ST0003,0.73,0.73,0.74,0.74,0.73,0.73,0.000,0.000,0.000,0.0,0.0,0.0,18.060,30.920,0.5654,4.2240,80.8559,125.04,12.1864,34.3816,56.6211,2.2500,0.00,1,0.00,-0.01,-0.01,0.00,0.00,0.000,0.000000,0.0,0.0
TR_000651952,2100-01-01,ST0004,1.01,1.01,1.01,1.01,1.01,1.01,0.000,0.000,0.000,0.0,0.0,0.0,0.029,0.000,0.0000,3.2098,252.5946,199.30,13.5951,2.7206,73.1707,2.1207,0.00,1,0.00,0.00,0.00,0.00,0.00,0.000,0.000000,0.0,0.0
TR_000328380,2100-01-01,ST0005,1.01,1.01,1.01,1.01,1.03,1.06,3.018,7.493,7.497,0.0,0.0,0.0,6.898,21.500,0.0811,3.6470,118.5237,246.72,17.1248,0.4779,88.0002,2.0000,0.03,1,0.00,0.00,0.00,-0.02,-0.05,0.895,0.000222,0.0,0.0


In [8]:
# ============================================================
# 2. 신규 B 파생변수 생성
# ============================================================
# 파생변수 'rain_sfc_imerg_pct_diff' = sfc_1h_pct - imerg_1h_pct
# 파생변수 'rain_sfc_imerg_pct_join' = min(sfc_1h_pct, imerg_1h_pct)
# 파생변수 'radar_sfc_imerg_pct_diff' = radar_pct - mean(sfc_1h_pct, imerg_1h_pct)

# 백분위 기준: 각 Fold의 학습 데이터
# 검증 데이터: 학습 Fold의 백분위 기준 적용

# Fold별 학습 데이터의 경험적 분포를 기준으로 백분위 산출
# 검증 데이터에도 학습 Fold의 분포 기준을 동일하게 적용 (정보 누수 방지)
# 학습 데이터에 동일한 값이 없으면 작은 값들의 누적 비율을 부여 (보간 없음)
# 학습 최솟값 미만은 0, 최댓값 초과는 1로 처리
# 백분위 범위: 0~1

def add_b_features(X_tr, X_va):

    X_tr = X_tr.copy()
    X_va = X_va.copy()

    pct_tr = {}
    pct_va = {}

    # 강우 관련 원본 컬럼
    rain_cols = [
        'sfc_rain_1h',
        'imerg_rain_1h',
        'radar_reflectivity_mean'
    ]

    # Fold 학습 데이터 기준 백분위 변환
    for col in rain_cols:

        ref = np.sort(X_tr[col].dropna().to_numpy())

        if len(ref) == 0:
            raise ValueError(f'{col}: 학습 Fold에 유효값이 없습니다.')

        for X, pct_dict in [(X_tr, pct_tr), (X_va, pct_va)]:

            values = X[col].to_numpy()

            pct = (
                np.searchsorted(ref, values, side='left') +
                np.searchsorted(ref, values, side='right')
            ) / (2 * len(ref))

            pct[pd.isna(values)] = np.nan

            pct_dict[col] = pct

    # 신규 파생변수 3개 생성
    for X, pct in [(X_tr, pct_tr), (X_va, pct_va)]:

        # B-1: 지상강우와 IMERG 백분위 차이
        X['rain_sfc_imerg_pct_diff'] = (
            pct['sfc_rain_1h'] -
            pct['imerg_rain_1h']
        )

        # B-2: 두 강우원의 공통 강우강도
        X['rain_sfc_imerg_pct_join'] = np.minimum(
            pct['sfc_rain_1h'],
            pct['imerg_rain_1h']
        )

        # B-3: 레이더와 다른 강우원의 백분위 차이
        X['radar_sfc_imerg_pct_diff'] = (
            pct['radar_reflectivity_mean'] -
            (
                pct['sfc_rain_1h'] +
                pct['imerg_rain_1h']
            ) / 2
        )

    return X_tr, X_va


# 신규 파생변수 목록
new_feature_cols_B = [
    'rain_sfc_imerg_pct_diff',
    'rain_sfc_imerg_pct_join',
    'radar_sfc_imerg_pct_diff'
]

# 기존 모델 입력변수에 추가
num_cols_fe_B = num_cols_fe + new_feature_cols_B

In [ ]:
# ============================================================
# 4. 신규 B 모델 학습용 데이터 준비
# ============================================================

# 기존 입력변수 구성 (B 파생변수는 Fold별로 생성)
X_train_with_station = train[
    num_cols_fe + ['station_id']
].copy()

# CatBoost 범주형 변수 처리
X_train_with_station['station_id'] = (
    X_train_with_station['station_id'].astype(str)
)

# 고정 Fold 매핑 파일 불러오기
kf_fold_fixed = pd.read_csv(
    PROJECT_ROOT / 'config' / '0_kf_event_fold_match.csv'
)

# event_id 기준 Fold 번호 연결 및 검증
print(train['event_id'].map(
    kf_fold_fixed.set_index('event_id')['fold_num']
).value_counts().sort_index())

# 기존 OOF와 고정 Fold 배정 비교
print('Fold 불일치:', (
    train_kf_oof['fold_num'].reindex(train.index)
    != train['event_id'].map(
        kf_fold_fixed.set_index('event_id')['fold_num']
    )
).sum())

event_id
1    122996
2    123399
3    123399
4    123216
5    123283
Name: count, dtype: int64
Fold 불일치: 0


In [12]:
# ============================================================
# 5. 신규 B: 고정 5-Fold 교차검증 및 모델 학습
# ============================================================

# 기존 입력변수 구성 (B 파생변수는 Fold별 생성)
X_train_with_station = train[
    num_cols_fe + ['station_id']
].copy()

# CatBoost 범주형 변수 처리
X_train_with_station['station_id'] = (
    X_train_with_station['station_id'].astype(str)
)

# 고정 Fold 매핑 파일 불러오기
kf_fold_fixed = pd.read_csv(
    PROJECT_ROOT / 'config' / '0_kf_event_fold_match.csv'
)

kf_results_B = []
kf_oof_pred_B = np.full(len(train), np.nan)

# 기존 Fold 배정 사용
kf_fold_num = train['event_id'].map(
    kf_fold_fixed.set_index('event_id')['fold_num']
)

assert kf_fold_num.notna().all(), 'Fold 배정 누락 발생'
assert (kf_fold_num.to_numpy() ==
        train_kf_oof['fold_num'].reindex(train.index).to_numpy()).all(), '기존 OOF와 Fold 불일치'

for fold_num in sorted(kf_fold_num.unique()):

    # 고정 Fold 기준 학습·검증 데이터 분할
    train_idx = np.flatnonzero(kf_fold_num.to_numpy() != fold_num)
    val_idx = np.flatnonzero(kf_fold_num.to_numpy() == fold_num)

    X_kf_tr = X_train_with_station.iloc[train_idx]
    X_kf_va = X_train_with_station.iloc[val_idx]

    y_kf_tr = y_train.iloc[train_idx]
    y_kf_va = y_train.iloc[val_idx]

    # Fold 학습 데이터 기준 신규 B 파생변수 생성
    X_kf_tr, X_kf_va = add_b_features(
        X_kf_tr, X_kf_va
    )

    # 신규 변수 포함 여부 확인
    assert X_kf_tr[new_feature_cols_B].shape[1] == 3
    assert X_kf_va[new_feature_cols_B].shape[1] == 3

    # CatBoost 모델 정의
    model_kf_B = CatBoostRegressor(
        iterations=3000,
        random_state=42,
        verbose=False
    )

    # 모델 학습
    model_kf_B.fit(
        X_kf_tr,
        y_kf_tr,
        cat_features=['station_id'],
        eval_set=(X_kf_va, y_kf_va),
        early_stopping_rounds=100,
        verbose=False
    )

    # OOF 예측
    pred_kf_B = model_kf_B.predict(X_kf_va)
    kf_oof_pred_B[val_idx] = pred_kf_B

    # Fold별 RMSE
    rmse_kf_B = root_mean_squared_error(
        y_kf_va,
        pred_kf_B
    )

    kf_results_B.append({
        'fold_num': fold_num,
        'rmse': rmse_kf_B,
        'best_iteration': model_kf_B.best_iteration_
    })

    print(f'Fold {fold_num}: RMSE = {rmse_kf_B:.6f}')

# 전체 OOF RMSE
assert np.isfinite(kf_oof_pred_B).all(), 'OOF 예측 누락 발생'

rmse_oof_B = root_mean_squared_error(
    y_train,
    kf_oof_pred_B
)

print('\n신규 B OOF RMSE:', rmse_oof_B)
print('기존 OOF RMSE:', 0.232826879)
print('RMSE 변화:', rmse_oof_B - 0.232826879)

display(pd.DataFrame(kf_results_B))

Fold 1: RMSE = 0.273449
Fold 2: RMSE = 0.147041
Fold 3: RMSE = 0.226527
Fold 4: RMSE = 0.273099
Fold 5: RMSE = 0.219639

신규 B OOF RMSE: 0.23256916235325611
기존 OOF RMSE: 0.232826879
RMSE 변화: -0.00025771664674387185


,fold_num,rmse,best_iteration
0,1,0.273449,907
1,2,0.147041,1747
2,3,0.226527,2961
3,4,0.273099,758
4,5,0.219639,1789


In [13]:
# ============================================================
# 7. 신규 B 적용 전후 강우사상별 SSE 비교
# ============================================================

# 신규 B 예측값과 기존 OOF의 row_id 순서 일치
assert train.index.equals(X_train_with_station.index)
assert train.index.is_unique
assert train_kf_oof.index.is_unique

# 강우사상별 SSE 비교
event_sse_B = (
    pd.DataFrame({
        'event_id': train['event_id'],
        'sse_base': train_kf_oof['resid_sq'].reindex(train.index),
        'sse_B': (y_train.to_numpy() - kf_oof_pred_B) ** 2
    })
    .groupby('event_id')
    .agg(
        count=('sse_base', 'size'),
        sse_base=('sse_base', 'sum'),
        sse_B=('sse_B', 'sum')
    )
)

# SSE 감소량: 양수일수록 개선
event_sse_B['delta_sse'] = (
    event_sse_B['sse_base'] - event_sse_B['sse_B']
)

# SSE 개선율(%)
event_sse_B['improve_pct'] = (
    event_sse_B['delta_sse'] / event_sse_B['sse_base'] * 100
)

# 기존 SSE가 큰 상위 6개 강우사상 비교
display(
    event_sse_B
    .sort_values('sse_base', ascending=False)
    .head(6)
    .round(3)
)

# 전체 강우사상별 SSE 개선 현황
print('SSE 개선 사상 수:', (event_sse_B['delta_sse'] > 0).sum())
print('SSE 악화 사상 수:', (event_sse_B['delta_sse'] < 0).sum())
print('전체 SSE 감소량:', event_sse_B['delta_sse'].sum())

,count,sse_base,sse_B,delta_sse,improve_pct
event_id,,,,,
98,71567,7696.683,7774.845,-78.162,-1.016
90,24970,3795.924,3788.259,7.665,0.202
35,19967,3318.869,3247.760,71.109,2.143
97,7408,3212.074,3209.111,2.963,0.092
3,18059,2627.781,2625.236,2.545,0.097
89,25289,2286.620,2280.704,5.916,0.259


SSE 개선 사상 수: 59
SSE 악화 사상 수: 39
전체 SSE 감소량: 73.91834054606656


In [ ]:
# 잠정 결론: 신규 B 변수 3개의 개선 효과는 A보다도 작다. 
# 강우 관측원 간 백분위 차이만으로는 기존의 고오차 현상을 충분히 설명하지 못하는 것으로 판단된다. 
# 전체 98개 사상에서의 개선 분포와 코드 정합성을 확인한 후 채택을 보류하는 것이 적절하다.

In [14]:
# ============================================================
# 8. 신규 B 모델 검증
# ============================================================

# 1. 데이터 행 및 Fold 배정 일치
assert train.index.equals(X_train_with_station.index)
assert train.index.equals(y_train.index)
assert train.index.equals(train_kf_oof.index)
assert train.index.is_unique

assert (
    kf_fold_num.to_numpy() ==
    train_kf_oof['fold_num'].to_numpy()
).all()

print('1. 데이터 행 및 Fold 배정: 정상')

# 2. 기존 OOF와 신규 B의 RMSE 검증
rmse_base_check = np.sqrt(
    train_kf_oof['resid_sq'].mean()
)

rmse_B_check = np.sqrt(
    np.mean((y_train.to_numpy() - kf_oof_pred_B) ** 2)
)

assert np.isclose(rmse_base_check, 0.232826879)
assert np.isclose(rmse_B_check, rmse_oof_B)

print('2. RMSE 계산: 정상')

# 3. 강우사상별 SSE 합계 검증
assert np.isclose(
    event_sse_B['sse_base'].sum(),
    train_kf_oof['resid_sq'].sum()
)

assert np.isclose(
    event_sse_B['sse_B'].sum(),
    np.sum((y_train.to_numpy() - kf_oof_pred_B) ** 2)
)

assert len(event_sse_B) == 98
assert event_sse_B['count'].sum() == len(train)

print('3. 강우사상별 SSE 계산: 정상')

# 4. 전체 강우사상별 개선 현황
print('\nSSE 개선 사상 수:', (event_sse_B['delta_sse'] > 0).sum())
print('SSE 악화 사상 수:', (event_sse_B['delta_sse'] < 0).sum())
print('전체 SSE 감소량:', event_sse_B['delta_sse'].sum())

# 5. 특정 강우사상에 대한 개선 집중도
display(
    event_sse_B
    .sort_values('delta_sse', ascending=False)
    .head(10)
    .round(3)
)

1. 데이터 행 및 Fold 배정: 정상
2. RMSE 계산: 정상
3. 강우사상별 SSE 계산: 정상

SSE 개선 사상 수: 59
SSE 악화 사상 수: 39
전체 SSE 감소량: 73.91834054606656


,count,sse_base,sse_B,delta_sse,improve_pct
event_id,,,,,
35,19967,3318.869,3247.760,71.109,2.143
96,19530,810.445,794.723,15.722,1.940
29,4185,421.442,405.957,15.485,3.674
94,8666,367.681,354.728,12.953,3.523
42,8030,188.335,179.068,9.267,4.920
90,24970,3795.924,3788.259,7.665,0.202
28,28931,939.045,931.865,7.180,0.765
47,16551,655.546,648.717,6.829,1.042
89,25289,2286.620,2280.704,5.916,0.259
